## The Iwasawa $\lambda$-invariant $\lambda_p(\chi)$ for arbitary Dirichlet characters  $\chi$: use special values of $L_p(s,\chi)$ to distinguish between the cases $\lambda = 0$, $\lambda = 1$, $\lambda = 2$ and $\lambda \geq 3$.

### Preprint: *Special values of $p$-adic $L$-functions and Iwasawa λ-invariants of Dirichlet characters* by Heiko Knospe.

In [1]:
def lambda_rk1(chi, p, prec=30):
    r"""
    Computes the lambda-invariant of the Iwasawa power series of chi * omega^{-1}
    in the rank-1 case (i.e. when chi(p) == 1).
    
    Uses the p-th Bernoulli number to detect lambda = 1.
    For lambda > 1 distinguishes between lambda = 2 and lambda >= 3
    via the derivative of L_p(s, chi) at s = 0.
    
    INPUT:
    
    - ``chi`` -- primitive odd Dirichlet character of the first kind
                 with chi(p) == 1
    
    - ``p`` -- odd prime
    
    - ``prec`` -- p-adic precision (default 30)
    
    OUTPUT: string "lambda=1", "lambda=2" or "lambda>=3"
    """
    if p == 2 or not p.is_prime():
        raise ValueError("p must be an odd prime number.")
    
    if not chi.is_primitive():
        raise ValueError("chi must be a primitive character.")
    
    if chi.is_even():
        print("The p-adic L-function is zero (even character).")
        return None
    
    if chi(p) != 1:
        raise ValueError("Not a rank-1 case (chi(p) != 1). Use lambda_rk0 instead.")
    
    cond = chi.conductor()
    if gcd(cond, p*p) == p*p:
        raise ValueError("chi must be a character of the first kind.")
    
    ord = chi.order()
    if gcd(ord, p) > 1:
        raise NotImplementedError("Characters of order divisible by p are not yet implemented.")
    
    f = mod(p, ord).multiplicative_order()
    
    K = Qp(p, prec)
    L.<a> = Qq(p**f, prec=prec)          # values of chi
    v = L.valuation()
    
    zeta = L.roots_of_unity(ord)[1]
    
    # p-th Bernoulli number
    bernp = (chi.bernoulli(p) / p).lift()(zeta)
    val = v(bernp)
    
    if val == 1:
        return "lambda=1"
    
    # Case lambda > 1: Derivation of L_p at s=0
    s = L(0)
    for a in range(cond):
        x = K(a / cond)
        s += chi(a).lift()(zeta) * log(gamma(x))
    
    a1 = s / log(K(1 + p))
    b = K((1 + p)**(1 - p) - 1)
    
    if v(-K(1 - p**(p-1)) * bernp - a1 * b) <= 2:
        return "lambda=2"
    else:
        return "lambda>=3"

In [2]:
# Example: conductor 19, p=7, order=6, odd character, twist=1
p=7

print(f'p={p}')

H = DirichletGroup(19, base_ring=CyclotomicField(6))

for i in range(H.order()):
    if H[i].is_primitive() and H[i].order()==6 and H[i].is_odd()==True and H[i](p)==1:
        chi = H[i]
        
        print(f'{chi} ==> {lambda_rk1(chi,p)}')
# two characters, lambda=2 and lambda=4 (>=3), same result as with other methods.

p=7
Dirichlet character modulo 19 of conductor 19 mapping 2 |--> zeta6 ==> lambda>=3
Dirichlet character modulo 19 of conductor 19 mapping 2 |--> -zeta6 + 1 ==> lambda=2


In [3]:
p=3

print(f'p={p}')

for n in range(1,240):
            H = DirichletGroup(n, base_ring=CyclotomicField(2))
            for i in range(H.order()):
                if H[i].order()==2 and H[i].is_odd()==True and H[i].is_primitive() and H[i](p)==1:
                    chi = H[i]
                    print(f'{chi} ==> {lambda_rk1(chi,p)}')
                    
# same result as with other methods.                    

p=3
Dirichlet character modulo 8 of conductor 8 mapping 7 |--> -1, 5 |--> -1 ==> lambda=1
Dirichlet character modulo 11 of conductor 11 mapping 2 |--> -1 ==> lambda=1
Dirichlet character modulo 20 of conductor 20 mapping 11 |--> -1, 17 |--> -1 ==> lambda=1
Dirichlet character modulo 23 of conductor 23 mapping 5 |--> -1 ==> lambda=1
Dirichlet character modulo 35 of conductor 35 mapping 22 |--> -1, 31 |--> -1 ==> lambda=2
Dirichlet character modulo 47 of conductor 47 mapping 5 |--> -1 ==> lambda=2
Dirichlet character modulo 56 of conductor 56 mapping 15 |--> 1, 29 |--> -1, 17 |--> -1 ==> lambda=2
Dirichlet character modulo 59 of conductor 59 mapping 2 |--> -1 ==> lambda=1
Dirichlet character modulo 68 of conductor 68 mapping 35 |--> -1, 37 |--> -1 ==> lambda=1
Dirichlet character modulo 71 of conductor 71 mapping 7 |--> -1 ==> lambda=1
Dirichlet character modulo 83 of conductor 83 mapping 2 |--> -1 ==> lambda=1
Dirichlet character modulo 95 of conductor 95 mapping 77 |--> -1, 21 |--> -1 

In [4]:
def lambda_rk0(chi, p, prec=30):
    r"""
    Computes the lambda-invariant of the Iwasawa power series of chi * omega^{-1}
    in the rank-0 case.
    
    Uses B_1,chi to detect lambda = 0,
    then B_p,chi to distinguish lambda = 1,
    and finally the derivative of L_p at s=0 for lambda = 2 vs. >= 3.
    
    INPUT:
    
    - ``chi`` -- primitive odd Dirichlet character of the first kind
    
    - ``p`` -- odd prime
    
    - ``prec`` -- p-adic precision (default 30)
    
    OUTPUT: string "lambda=0", "lambda=1", "lambda=2" or "lambda>=3"
    """
    if p == 2 or not p.is_prime():
        raise ValueError("p must be an odd prime number.")
    
    if not chi.is_primitive():
        raise ValueError("chi must be a primitive character.")
    
    if chi.is_even():
        print("The p-adic L-function is zero (even character).")
        return None
    
    cond = chi.conductor()
    if gcd(cond, p*p) == p*p:
        raise ValueError("chi must be a character of the first kind.")
    
    ord = chi.order()
    if gcd(ord, p) > 1:
        raise NotImplementedError("Characters of order divisible by p are not yet implemented.")
    
    f = mod(p, ord).multiplicative_order()
    
    K = Qp(p, prec)
    L.<a> = Qq(p**f, prec=prec)
    v = L.valuation()
    
    zeta = L.roots_of_unity(ord)[1]
    
    # First Bernoulli number → lambda = 0 ?
    bern1 = chi.bernoulli(1).lift()(zeta)
    if v(bern1) == 0:
        return "lambda=0"
    
    # Fall lambda > 0
    chip = chi(p).lift()(zeta)
    a0 = -(1 - chip) * bern1
    
    bernp = (chi.bernoulli(p) / p).lift()(zeta)
    val = v(-(1 - chip * p**(p-1)) * bernp - a0)
    
    if val == 1:
        return "lambda=1"
    
    # Case lambda > 1: Derivative of L_p at s=0
    s = L(0)
    
    if gcd(cond, p) == 1:
        for a in range(cond):
            x = K(a / cond)
            s += chi(a).lift()(zeta) * log(gamma(x))
    else:
        for a in range(1, cond):
            if gcd(a, p) == 1:
                x = K(a / cond)
                # Approximation of digamma-Funktion / log-gamma
                s += chi(a).lift()(zeta) * ((x - 1/2)*x.log(p_branch=0)
                                           - x
                                           + (1/12)/x
                                           - (1/30)*(1/12)/x**3)
    
    s += (1 - chip) * K(cond).log(p_branch=0) * bern1
    a1 = s / log(K(1 + p))
    b = K((1 + p)**(1 - p) - 1)
    
    c = -(1 - chip * p**(p-1)) * bernp - a1 * b - a0
    
    if v(c) <= 2:
        return "lambda=2"
    else:
        return "lambda>=3"

In [5]:
p=3

print(f'p={p}')

# order 8 characters chi

for n in range(1,200):
            H = DirichletGroup(n, base_ring=CyclotomicField(8))
            for i in range(H.order()):
                if H[i].order()==8 and H[i].is_odd()==True and H[i].is_primitive() and H[i](p)!=1:
                    chi = H[i]
                    print(f'{chi} ==> {lambda_rk0(chi,p)}')
# same result as with other methods.                     

p=3
Dirichlet character modulo 32 of conductor 32 mapping 31 |--> -1, 5 |--> zeta8 ==> lambda=0
Dirichlet character modulo 32 of conductor 32 mapping 31 |--> -1, 5 |--> zeta8^3 ==> lambda=0
Dirichlet character modulo 32 of conductor 32 mapping 31 |--> -1, 5 |--> -zeta8 ==> lambda=0
Dirichlet character modulo 32 of conductor 32 mapping 31 |--> -1, 5 |--> -zeta8^3 ==> lambda=0
Dirichlet character modulo 41 of conductor 41 mapping 6 |--> zeta8 ==> lambda=0
Dirichlet character modulo 41 of conductor 41 mapping 6 |--> zeta8^3 ==> lambda=0
Dirichlet character modulo 41 of conductor 41 mapping 6 |--> -zeta8 ==> lambda=0
Dirichlet character modulo 41 of conductor 41 mapping 6 |--> -zeta8^3 ==> lambda=0
Dirichlet character modulo 51 of conductor 51 mapping 35 |--> -1, 37 |--> zeta8 ==> lambda=0
Dirichlet character modulo 51 of conductor 51 mapping 35 |--> -1, 37 |--> zeta8^3 ==> lambda=0
Dirichlet character modulo 51 of conductor 51 mapping 35 |--> -1, 37 |--> -zeta8 ==> lambda=0
Dirichlet char